# Synthesis

The synthesis stage is responsible for
- Finding the optimal units for a target phoneme sequence via Viterbi search
- Concatenating these units either via raw concatenation or TPSOLA

In [19]:
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from pathlib import Path

In [20]:
# Phonological feature lookup table (from features.ipynb)
MANNERS = ['vowel', 'plosive', 'fricative', 'affricate', 'nasal', 'liquid', 'glide', 'silence']
PLACES  = ['bilabial', 'labiodental', 'dental', 'alveolar', 'postalveolar', 'palatal', 'velar', 'glottal', 'none']

PHONEME_FEATURES = {
    # Silence / Pauses
    'pau' : {'is_vowel': 0, 'is_voiced': 0, 'manner': 'silence', 'place': 'none'},
    'ssil': {'is_vowel': 0, 'is_voiced': 0, 'manner': 'silence', 'place': 'none'},
    # Vowels (all voiced)
    'aa' : {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ae' : {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ah' : {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ao' : {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'aw' : {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ax' : {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ay' : {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'eh' : {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'er' : {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ey' : {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ih' : {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'iy' : {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ow' : {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'oy' : {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'uh' : {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'uw' : {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    # Stops / Plosives
    'p'  : {'is_vowel': 0, 'is_voiced': 0, 'manner': 'plosive', 'place': 'bilabial'},
    'b'  : {'is_vowel': 0, 'is_voiced': 1, 'manner': 'plosive', 'place': 'bilabial'},
    't'  : {'is_vowel': 0, 'is_voiced': 0, 'manner': 'plosive', 'place': 'alveolar'},
    'd'  : {'is_vowel': 0, 'is_voiced': 1, 'manner': 'plosive', 'place': 'alveolar'},
    'k'  : {'is_vowel': 0, 'is_voiced': 0, 'manner': 'plosive', 'place': 'velar'},
    'g'  : {'is_vowel': 0, 'is_voiced': 1, 'manner': 'plosive', 'place': 'velar'},
    # Fricatives
    'f'  : {'is_vowel': 0, 'is_voiced': 0, 'manner': 'fricative', 'place': 'labiodental'},
    'v'  : {'is_vowel': 0, 'is_voiced': 1, 'manner': 'fricative', 'place': 'labiodental'},
    'th' : {'is_vowel': 0, 'is_voiced': 0, 'manner': 'fricative', 'place': 'dental'},
    'dh' : {'is_vowel': 0, 'is_voiced': 1, 'manner': 'fricative', 'place': 'dental'},
    's'  : {'is_vowel': 0, 'is_voiced': 0, 'manner': 'fricative', 'place': 'alveolar'},
    'z'  : {'is_vowel': 0, 'is_voiced': 1, 'manner': 'fricative', 'place': 'alveolar'},
    'sh' : {'is_vowel': 0, 'is_voiced': 0, 'manner': 'fricative', 'place': 'postalveolar'},
    'zh' : {'is_vowel': 0, 'is_voiced': 1, 'manner': 'fricative', 'place': 'postalveolar'},
    'hh' : {'is_vowel': 0, 'is_voiced': 0, 'manner': 'fricative', 'place': 'glottal'},
    # Affricates
    'ch' : {'is_vowel': 0, 'is_voiced': 0, 'manner': 'affricate', 'place': 'postalveolar'},
    'jh' : {'is_vowel': 0, 'is_voiced': 1, 'manner': 'affricate', 'place': 'postalveolar'},
    # Nasals
    'm'  : {'is_vowel': 0, 'is_voiced': 1, 'manner': 'nasal', 'place': 'bilabial'},
    'n'  : {'is_vowel': 0, 'is_voiced': 1, 'manner': 'nasal', 'place': 'alveolar'},
    'ng' : {'is_vowel': 0, 'is_voiced': 1, 'manner': 'nasal', 'place': 'velar'},
    # Liquids & Glides
    'l'  : {'is_vowel': 0, 'is_voiced': 1, 'manner': 'liquid', 'place': 'alveolar'},
    'r'  : {'is_vowel': 0, 'is_voiced': 1, 'manner': 'liquid', 'place': 'alveolar'},
    'w'  : {'is_vowel': 0, 'is_voiced': 1, 'manner': 'glide', 'place': 'bilabial'},
    'y'  : {'is_vowel': 0, 'is_voiced': 1, 'manner': 'glide', 'place': 'palatal'},
}

In [21]:
# ---------------------------------------------------------------------------
# 1. Format the target phoneme sequence into a feature DataFrame
# ---------------------------------------------------------------------------

def format_target_seq(phoneme_seq):
    """
    Convert a list of phoneme strings (target utterance) into a DataFrame
    with phonological features and context for each position.

    Parameters
    ----------
    phoneme_seq : list of str
        Target phoneme sequence, e.g. ['hh', 'iy', 'd']

    Returns
    -------
    pd.DataFrame with columns:
        ph, nph, pph, is_vowel, is_voiced, manner, place, dur (default 0)
    """
    rows = []
    n = len(phoneme_seq)
    for i, ph in enumerate(phoneme_seq):
        feats = PHONEME_FEATURES.get(ph, {'is_vowel': 0, 'is_voiced': 0,
                                           'manner': 'silence', 'place': 'none'})
        pph = phoneme_seq[i - 1] if i > 0 else 'pau'
        nph = phoneme_seq[i + 1] if i < n - 1 else 'pau'
        rows.append({
            'ph': ph,
            'nph': nph,
            'pph': pph,
            'is_vowel': feats['is_vowel'],
            'is_voiced': feats['is_voiced'],
            'manner': feats['manner'],
            'place': feats['place'],
            'dur': 0.0,   # unknown at synthesis time
        })
    return pd.DataFrame(rows)

In [22]:
# ---------------------------------------------------------------------------
# 2. Cost functions
# ---------------------------------------------------------------------------

def _binary_diff(target_val, candidate_val):
    """0 if same, 1 if different."""
    return 0.0 if target_val == candidate_val else 1.0


def _abs_diff(a, b):
    """Absolute difference."""
    return float(abs(a - b))


# Target sub-cost computation using vectorised ops where possible.
#
# We separate costs into two groups:
#   a) categorical/binary fields compared with equality
#   b) scalar fields compared with absolute difference  (none known at synth time)

TARGET_BINARY_FIELDS = ['nph', 'pph', 'is_vowel', 'is_voiced', 'manner', 'place']


def compute_target_costs(target_row, cand_df, weights):
    """
    Return a numpy array of target costs for every row in cand_df.
    """
    cost = np.zeros(len(cand_df), dtype=np.float64)
    for field in TARGET_BINARY_FIELDS:
        w = weights.get(field, 1.0)
        cost += w * (cand_df[field].values != target_row[field]).astype(np.float64)
    return cost

In [23]:
# ---------------------------------------------------------------------------
# 3. Concatenation costs (fully vectorised)
# ---------------------------------------------------------------------------

def compute_concat_costs(prev_cands, curr_cands, weights):
    """
    Return shape (n_prev, n_curr) matrix of concatenation costs.

    Each entry (j, k) is the cost of concatenating prev_cands[j]
    followed by curr_cands[k].
    """
    n_prev = len(prev_cands)
    n_curr = len(curr_cands)

    cost = np.zeros((n_prev, n_curr), dtype=np.float64)

    # MFCC distance (Euclidean between all pairs)
    w_mfcc = weights.get('mfcc', 1.0)
    prev_mfcc = np.array(prev_cands['mfcc'].tolist(), dtype=np.float64)  # (n_prev, D)
    curr_mfcc = np.array(curr_cands['mfcc'].tolist(), dtype=np.float64)  # (n_curr, D)
    # ||a - b||^2 = ||a||^2 + ||b||^2 - 2 * a^T b
    # Using broadcasting for all-pairs squared Euclidean:
    # shape (n_prev, n_curr)
    sq_prev = np.sum(prev_mfcc ** 2, axis=1, keepdims=True)   # (n_prev, 1)
    sq_curr = np.sum(curr_mfcc ** 2, axis=1, keepdims=True)   # (n_curr, 1)
    cross   = prev_mfcc @ curr_mfcc.T                          # (n_prev, n_curr)
    mfcc_dist = np.sqrt(np.maximum(0.0, sq_prev + sq_curr.T - 2.0 * cross))
    cost += w_mfcc * mfcc_dist

    # Log-power absolute difference
    w_pow = weights.get('log_power', 0.5)
    pow_prev = prev_cands['log_power'].values.astype(np.float64).reshape(-1, 1)  # (n_prev, 1)
    pow_curr = curr_cands['log_power'].values.astype(np.float64).reshape(1, -1)  # (1, n_curr)
    cost += w_pow * np.abs(pow_prev - pow_curr)

    # Log-pitch absolute difference
    w_pitch = weights.get('log_pitch', 0.5)
    pitch_prev = prev_cands['log_pitch'].values.astype(np.float64).reshape(-1, 1)
    pitch_curr = curr_cands['log_pitch'].values.astype(np.float64).reshape(1, -1)
    cost += w_pitch * np.abs(pitch_prev - pitch_curr)

    return cost

In [24]:
# ---------------------------------------------------------------------------
# 4. Database loading helpers
# ---------------------------------------------------------------------------

WAV_ROOT = Path('datasets/cmu_us_awb_arctic')


def load_all_candidates(db_path, target_df, top_n=50):
    """
    Load candidate units from the database for every unique phoneme
    appearing in the target sequence.

    For each phoneme, we keep the *top_n* candidates with the best
    (lowest) target cost against the corresponding target position.
    This prunes the search space while retaining high-quality units.

    Parameters
    ----------
    db_path : str or Path
    target_df : pd.DataFrame
        Output of format_target_seq().
    top_n : int
        Maximum candidates to keep per position.

    Returns
    -------
    candidates_per_pos : list of pd.DataFrame
        One DataFrame per target position, each with at most top_n rows.
    """
    # Pre-load full database once (avoids repeated parquet reads)
    full_db = pd.read_parquet(db_path)

    unique_phs = set(target_df['ph'].unique())
    cache = {ph: full_db[full_db['ph'] == ph].reset_index(drop=True)
             for ph in unique_phs}

    candidates_per_pos = []
    for i in range(len(target_df)):
        ph = target_df.iloc[i]['ph']
        cands = cache.get(ph)
        if cands is None or len(cands) == 0:
            raise ValueError(f'No candidates found for phoneme "{ph}"')

        if top_n is not None and len(cands) > top_n:
            # Prune to top_n by target cost
            t_costs = compute_target_costs(target_df.iloc[i], cands,
                                           _default_weights())
            best_idx = np.argpartition(t_costs, top_n)[:top_n]
            cands = cands.iloc[best_idx].reset_index(drop=True)

        candidates_per_pos.append(cands)

    return candidates_per_pos


def _default_weights():
    """Default weight dictionary used when none is provided."""
    return {
        # Target-cost weights (learned from training.ipynb)
        'nph':       9.62,
        'pph':       9.71,
        'is_vowel':  5.0,
        'is_voiced': 5.0,
        'manner':    5.0,
        'place':     5.0,
        # Concatenation-cost weights
        'mfcc':      1.0,
        'log_power': 46,
        'log_pitch': 25,
    }

In [25]:
# ---------------------------------------------------------------------------
# 5. Viterbi search
# ---------------------------------------------------------------------------

def viterbi(target_seq, db_path, weights=None, top_n=50):
    """
    Find the optimal sequence of database units for a target phoneme
    sequence using Viterbi search.

    Parameters
    ----------
    target_seq : list of str
        Target phoneme sequence, e.g. ['hh', 'iy', 'd'].
    db_path : str or Path
        Path to the feature database Parquet file.
    weights : dict, optional
        Weight dictionary for sub-costs.  Defaults to the learned weights
        from training.ipynb plus sensible defaults for concatenation.
    top_n : int
        Maximum candidates to retain per position (pruning by target cost).

    Returns
    -------
    best_path : list of pandas.Series
        The selected database rows (one per target position) forming
        the minimum-cost path.
    total_cost : float
        Cumulative cost of the best path.
    """
    if weights is None:
        weights = _default_weights()

    # --- 1. Build target feature table ---
    target_df = format_target_seq(target_seq)
    n_pos = len(target_df)

    # --- 2. Load + prune candidates per position ---
    candidates_per_pos = load_all_candidates(db_path, target_df, top_n=top_n)

    # --- 3. Pre-compute target costs for each position ---
    target_costs = []
    for i in range(n_pos):
        t_costs = compute_target_costs(
            target_df.iloc[i], candidates_per_pos[i], weights)
        target_costs.append(t_costs)

    # --- 4. Viterbi DP (fully vectorised concatenation costs) ---
    dp = [None] * n_pos
    bp = [None] * n_pos

    # Initialise position 0
    dp[0] = target_costs[0].copy()
    bp[0] = np.full(len(candidates_per_pos[0]), -1, dtype=np.int32)

    for i in range(1, n_pos):
        prev_cands = candidates_per_pos[i - 1]
        curr_cands = candidates_per_pos[i]

        # concat_costs shape: (n_prev, n_curr)
        concat_costs = compute_concat_costs(prev_cands, curr_cands, weights)

        # dp[i-1] is (n_prev,)  ->  broadcast to (n_prev, n_curr)
        total = dp[i - 1][:, np.newaxis] + concat_costs + target_costs[i][np.newaxis, :]

        # Best predecessor per current candidate
        best_j = np.argmin(total, axis=0)            # (n_curr,)
        dp[i] = total[best_j, np.arange(len(curr_cands))]
        bp[i] = best_j.astype(np.int32)

    # --- 5. Backtrack ---
    best_last = int(np.argmin(dp[n_pos - 1]))
    total_cost = float(dp[n_pos - 1][best_last])

    path_indices = [0] * n_pos
    path_indices[n_pos - 1] = best_last
    for i in range(n_pos - 2, -1, -1):
        path_indices[i] = int(bp[i + 1][path_indices[i + 1]])

    best_path = [
        candidates_per_pos[i].iloc[path_indices[i]]
        for i in range(n_pos)
    ]

    return best_path, total_cost

In [26]:
# ---------------------------------------------------------------------------
# 6. Waveform synthesis from selected units
# ---------------------------------------------------------------------------

def _resolve_wav_path(candidate_row):
    """
    Given a database row with an ``id`` like ``arctic_b0169_0000``,
    return the path to the corresponding .wav file.
    """
    # id format: {filestem}_{index:04d}
    file_stem = str(candidate_row['id']).rsplit('_', 1)[0]
    return WAV_ROOT / 'wav' / f'{file_stem}.wav'


def synthesize_wav(phonemes, fout_name):
    """
    Construct a waveform by concatenating audio segments from the
    selected candidate units.

    Parameters
    ----------
    phonemes : list of pandas.Series
        The selected candidate units (one per target phoneme).
        Each row must have at least: ``id``, ``start`` (seconds),
        ``dur`` (seconds).
    fout_name : str or Path
        Path where the output .wav file will be written.

    Returns
    -------
    fout_name : Path
    """
    out_buff = []
    sr = None

    for ph in phonemes:
        wav_path = _resolve_wav_path(ph)
        start = float(ph['start'])
        dur   = float(ph['dur'])

        samples, sr = librosa.load(str(wav_path), sr=None)
        seg = samples[int(start * sr) : int(start * sr + dur * sr)]
        out_buff.append(seg)

    if len(out_buff) == 0:
        raise ValueError('No audio segments to concatenate.')

    waveform = np.concatenate(out_buff)
    sf.write(str(fout_name), waveform, sr)
    return Path(fout_name)

In [27]:
# ---------------------------------------------------------------------------
# 7. End-to-end synthesis convenience
# ---------------------------------------------------------------------------

def synthesize(target_phonemes, db_path, fout_name, weights=None, top_n=50):
    """
    Run Viterbi unit selection followed by raw concatenation synthesis.

    Parameters
    ----------
    target_phonemes : list of str
    db_path : str or Path
    fout_name : str or Path
    weights : dict, optional
    top_n : int
        Maximum candidates to retain per position.

    Returns
    -------
    best_path, total_cost, out_path
    """
    best_path, total_cost = viterbi(target_phonemes, db_path,
                                    weights=weights, top_n=top_n)
    out_path = synthesize_wav(best_path, fout_name)
    return best_path, total_cost, out_path

In [28]:
synthesize(
["pau",
"pau",
"ao",
"th",
"er",
"ah",
"v",
"dh",
"ax",
"d",
"ey",
"n",
"jh",
"er",
"t",
"r",
"ey",
"l",
"pau",
"f",
"ih",
"l",
"ax",
"p",
"s",
"t",
"iy",
"l",
"z",
"eh",
"t",
"s",
"eh",
"t",
"er",
"ax",
"pau",], 'datasets/cmu_us_awb_arctic/feats/a0.parquet', 'author.wav')

([id                                           arctic_b0051_0000
  ph                                                         pau
  nph                                                         dh
  pph                                                        pau
  start                                                  0.13075
  dur                                                     0.1625
  is_vowel                                                     0
  is_voiced                                                    0
  manner                                                 silence
  place                                                     none
  log_power                                            -4.054866
  log_pitch                                             4.761517
  mfcc         [-569.3458251953125, 114.25843811035156, 20.94...
  Name: 18, dtype: object,
  id                                           arctic_b0051_0000
  ph                                                         pa